# Virginia Piedmont native phenology

This notebook prototypes a pyinaturalist-first workflow for seasonal prevalence in the Virginia Piedmont iNaturalist project.

Current assumptions:
- Region scope comes from the Virginia Physiographic Regions: Piedmont project.
- Nativity is evaluated against Virginia.
- Moths and butterflies are treated together as Lepidoptera.
- Life-stage slices are explicit for Lepidoptera: overall, larva, pupa, and adult.
- Bees (Anthophila) are included as a second focus group with an overall-only slice.

In [3]:
from pathlib import Path
import sys

import pandas as pd
import plotly.express as px

repo_root = Path.cwd()
if not (repo_root / 'helpers.py').exists() and (repo_root.parent.parent / 'helpers.py').exists():
    repo_root = repo_root.parent.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from helpers import annotate_taxon_nativity, get_inat_session, get_observation_rows

PROJECT_ID = 'virginia-physiographic-regions-piedmont'
VA_PLACE_ID = 1297
LEPIDOPTERA_TAXON_ID = 47157
ANTHOPHILA_TAXON_ID = 630955
LIFESTAGE_TERM_ID = 1
LIFESTAGE_ADULT = 2
LIFESTAGE_PUPA = 4
LIFESTAGE_LARVA = 6
START = '2023-01-01'
END = None
PER_PAGE = 200
MAX_PAGES = 30

session = get_inat_session()

In [ ]:
query_specs = [
    {
        'focus_group': 'Lepidoptera',
        'life_stage_bucket': 'overall',
        'taxa_filters': {'taxon_id': LEPIDOPTERA_TAXON_ID},
    },
    {
        'focus_group': 'Lepidoptera',
        'life_stage_bucket': 'larva',
        'taxa_filters': {
            'taxon_id': LEPIDOPTERA_TAXON_ID,
            'term_id': LIFESTAGE_TERM_ID,
            'term_value_id': LIFESTAGE_LARVA,
        },
    },
    {
        'focus_group': 'Lepidoptera',
        'life_stage_bucket': 'pupa',
        'taxa_filters': {
            'taxon_id': LEPIDOPTERA_TAXON_ID,
            'term_id': LIFESTAGE_TERM_ID,
            'term_value_id': LIFESTAGE_PUPA,
        },
    },
    {
        'focus_group': 'Lepidoptera',
        'life_stage_bucket': 'adult',
        'taxa_filters': {
            'taxon_id': LEPIDOPTERA_TAXON_ID,
            'term_id': LIFESTAGE_TERM_ID,
            'term_value_id': LIFESTAGE_ADULT,
        },
    },
    {
        'focus_group': 'Anthophila',
        'life_stage_bucket': 'overall',
        'taxa_filters': {'taxon_id': ANTHOPHILA_TAXON_ID},
    },
]

frames = []
for spec in query_specs:
    frame = get_observation_rows(
        kind='any',
        project_id=PROJECT_ID,
        taxa_filters=spec['taxa_filters'],
        d1=START,
        d2=END,
        per_page=PER_PAGE,
        max_pages=MAX_PAGES,
        session=session,
    )
    frame['focus_group'] = spec['focus_group']
    frame['life_stage_bucket'] = spec['life_stage_bucket']
    frames.append(frame)

phenology = pd.concat(frames, ignore_index=True)
phenology = annotate_taxon_nativity(phenology, nativity_place_id=VA_PLACE_ID, session=session)
native_phenology = phenology[phenology['nativity'].isin(['Native', 'Endemic'])].copy()
native_phenology['observed_on'] = pd.to_datetime(native_phenology['observed_on'])
native_phenology['month'] = native_phenology['observed_on'].dt.month
native_phenology['month_name'] = native_phenology['observed_on'].dt.strftime('%b')
native_phenology[
    ['obs_id', 'focus_group', 'taxon_name', 'taxon_preferred_common_name', 'life_stage_bucket', 'nativity', 'observed_on']
].head()

In [ ]:
monthly_taxa = (
    native_phenology.groupby(
        [
            'focus_group',
            'life_stage_bucket',
            'month',
            'month_name',
            'taxon_id',
            'taxon_name',
            'taxon_preferred_common_name',
        ],
        dropna=False,
    )['obs_id']
    .nunique()
    .rename('observation_count')
    .reset_index()
)

poster_candidates = (
    native_phenology.groupby(
        ['focus_group', 'life_stage_bucket', 'taxon_id', 'taxon_name', 'taxon_preferred_common_name', 'nativity'],
        dropna=False,
    )
    .agg(
        observation_count=('obs_id', 'nunique'),
        months_present=('month', 'nunique'),
        first_photo=('photo_url', 'first'),
    )
    .reset_index()
    .sort_values(
        ['focus_group', 'life_stage_bucket', 'observation_count', 'months_present'],
        ascending=[True, True, False, False],
    )
)

poster_candidates.head(30)

In [ ]:
top_taxa = poster_candidates.groupby(['focus_group', 'life_stage_bucket']).head(4)['taxon_id']
plot_frame = monthly_taxa[monthly_taxa['taxon_id'].isin(top_taxa)].copy()
month_order = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
stage_order = ['overall', 'adult', 'pupa', 'larva']
plot_frame['month_name'] = pd.Categorical(plot_frame['month_name'], categories=month_order, ordered=True)
plot_frame['life_stage_bucket'] = pd.Categorical(plot_frame['life_stage_bucket'], categories=stage_order, ordered=True)
plot_frame = plot_frame.sort_values(['focus_group', 'life_stage_bucket', 'taxon_name', 'month'])

fig = px.line(
    plot_frame,
    x='month_name',
    y='observation_count',
    color='taxon_preferred_common_name',
    facet_row='focus_group',
    facet_col='life_stage_bucket',
    markers=True,
    hover_data=['taxon_name'],
    title='Monthly prevalence of selected native Lepidoptera and Anthophila in the Virginia Piedmont',
)
fig.update_yaxes(matches=None)
fig.show()